In [1]:
import pandas as pd
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("suchintikasarkar/sentiment-analysis-for-mental-health")

print("Path to dataset files:", path)

os.listdir(path)

c:\Users\HopeE\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\HopeE\.cache\kagglehub\datasets\suchintikasarkar\sentiment-analysis-for-mental-health\versions\1


['Combined Data.csv']

In [2]:
import os

path_to_csv = os.path.join(path, "Combined Data.csv")

df = pd.read_csv(path_to_csv)

In [3]:
df = df[df['status'].isin(['Normal', 'Depression', 'Anxiety', 'Suicidal', 'Stress'])]

status
Normal                  16351
Depression              15404
Suicidal                10653
Anxiety                  3888
Bipolar                  2877
Stress                   2669
Personality disorder     1201
Name: count, dtype: int64

Restructure label into severity level.

In [5]:
df = df[df['status'].isin(['Normal', 'Depression', 'Anxiety', 'Suicidal', 'Stress'])]

# Convert status into severity levels
severity_mapping = {
    'Normal': 0,
    'Stress': 1,
    'Anxiety': 1,
    'Depression': 2,
    'Suicidal': 3
}

df['severity'] = df['status'].map(severity_mapping)

In [6]:
df.drop(columns=['status'], inplace=True)

In [8]:
# Rename statement to text
df.rename(columns={'statement': 'text'}, inplace=True)

In [9]:
df.head()

,Unnamed: 0,text,severity
0,0,oh my gosh,1
1,1,"trouble sleeping, confused mind, restless hear...",1
2,2,"All wrong, back off dear, forward doubt. Stay ...",1
3,3,I've shifted my focus to something else but I'...,1
4,4,"I'm restless and restless, it's been a month n...",1


## Preprocess Dataset

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['severity'],
    test_size=0.2,
    stratify=df['severity'],
    random_state=42
)

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=3,
    max_df=0.9
)

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

models = {
    "logreg": LogisticRegression(
        class_weight='balanced',
        max_iter=1000
    ),
    "svm": LinearSVC(class_weight='balanced'),
    "rf": RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced'
    )
}

### Evaluation

In [14]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro']

# Handle NaN values in X_train and align y_train
mask = X_train.notna()
X_train = X_train[mask]
y_train = y_train[mask]

results = {}

for name, model in models.items():
    pipeline = make_pipeline(tfidf, model)
    
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )
    
    results[name] = scores

In [17]:
results

{'logreg': {'fit_time': array([8.21875572, 8.39612675, 8.96437764, 7.92664933, 8.62958145]),
  'score_time': array([0.98985958, 1.00925517, 0.98213196, 1.00625467, 0.97165108]),
  'test_accuracy': array([0.78740561, 0.7999488 , 0.80120328, 0.7937788 , 0.80209933]),
  'test_f1_macro': array([0.7798859 , 0.79091134, 0.79164473, 0.7855752 , 0.79341764]),
  'test_precision_macro': array([0.77315099, 0.78570336, 0.78466341, 0.77944445, 0.78739658]),
  'test_recall_macro': array([0.79452232, 0.80139399, 0.80559907, 0.79813223, 0.80410923])},
 'svm': {'fit_time': array([9.25057912, 9.41629148, 9.40600371, 9.42422271, 9.4738698 ]),
  'score_time': array([0.96342683, 0.94259882, 0.95182991, 1.0259738 , 0.95436001]),
  'test_accuracy': array([0.78484577, 0.78689364, 0.79160266, 0.79006656, 0.79429083]),
  'test_f1_macro': array([0.77642082, 0.7774116 , 0.78046679, 0.77970939, 0.78457489]),
  'test_precision_macro': array([0.77440295, 0.77804007, 0.7772181 , 0.7765053 , 0.78309368]),
  'test_reca